In [6]:
import os
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np


from first_model import AMIVADDataset, VADLightningModule

# Import comprehensive evaluation metrics
from torchmetrics.classification import (
    BinaryAccuracy, 
    BinaryPrecision, 
    BinaryRecall, 
    BinaryF1Score, 
    BinaryAUROC, 
    BinaryConfusionMatrix
)

In [9]:

def evaluate_model(checkpoint_path, audio_dir, test_rttm_dir, batch_size=32, device='cuda'):
    # 1. Set up device
    if device == 'cuda' and not torch.cuda.is_available():
        print("CUDA not available. Falling back to CPU.")
        device = 'cpu'
    device = torch.device(device)
    print(f"Using device: {device}")

    # 2. Load the trained model from checkpoint
    print(f"Loading model from checkpoint: {checkpoint_path}")
    model = VADLightningModule.load_from_checkpoint(checkpoint_path)
    model.to(device)
    model.eval()  # Set model to evaluation mode (disables dropout/batchnorm updates)

    # 3. Prepare Test Dataset and DataLoader
    print("Preparing test dataset...")
    test_dataset = AMIVADDataset(audio_dir=audio_dir, rttm_dir=test_rttm_dir)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    print(f"Total test chunks to evaluate: {len(test_dataset)}")

    # 4. Initialize Metrics (all evaluated at frame-level)
    acc_metric = BinaryAccuracy().to(device)
    prec_metric = BinaryPrecision().to(device)
    rec_metric = BinaryRecall().to(device)
    f1_metric = BinaryF1Score().to(device)
    auroc_metric = BinaryAUROC().to(device)
    cm_metric = BinaryConfusionMatrix().to(device)

    # 5. Evaluation Loop
    print("\nRunning evaluation...")
    with torch.no_grad():  # Disable gradient calculation for faster inference & less memory
        for waveforms, labels in tqdm(test_loader, desc="Testing"):
            waveforms = waveforms.to(device)
            labels = labels.to(device)

            # Model forward pass
            logits = model(waveforms)

            # Align feature dimensions (exact same logic as training shared_step)
            if logits.shape[1] != labels.shape[1]:
                logits = torch.nn.functional.interpolate(
                    logits.unsqueeze(1), size=labels.shape[1], mode='linear'
                ).squeeze(1)

            # Get probabilities and hard binary predictions
            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).int()
            labels_int = labels.int()

            # Flatten tensors to evaluate frame-by-frame globally
            preds_flat = preds.view(-1)
            labels_flat = labels_int.view(-1)
            probs_flat = probs.view(-1)

            # Update running metrics
            acc_metric.update(preds_flat, labels_flat)
            prec_metric.update(preds_flat, labels_flat)
            rec_metric.update(preds_flat, labels_flat)
            f1_metric.update(preds_flat, labels_flat)
            auroc_metric.update(probs_flat, labels_flat)
            cm_metric.update(preds_flat, labels_flat)

    # 6. Compute Final Metrics
    print("\n" + "="*30)
    print("      VAD EVALUATION REPORT      ")
    print("="*30)
    print(f"Frame-level Accuracy:  {acc_metric.compute().item():.4f}")
    print(f"Precision (Speech):    {prec_metric.compute().item():.4f}")
    print(f"Recall (Sensitivity):  {rec_metric.compute().item():.4f}")
    print(f"F1-Score:              {f1_metric.compute().item():.4f}")
    print(f"AUROC:                 {auroc_metric.compute().item():.4f}")
    print("="*30)

    # 7. Plot and Save Confusion Matrix
    cm = cm_metric.compute().cpu().numpy()
    plot_confusion_matrix(cm, save_path='confusion_matrix.png')
    print("Confusion Matrix plot saved as 'confusion_matrix.png'")


def plot_confusion_matrix(cm, save_path='confusion_matrix.png'):
    """Helper function to plot a clean confusion matrix."""
    fig, ax = plt.subplots(figsize=(6, 6))
    # Normalize matrix rows to show percentages
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    im = ax.imshow(cm_norm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    classes = ['Non-Speech (0)', 'Speech (1)']
    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title='Normalized Confusion Matrix (Frame-level)',
           ylabel='True Label',
           xlabel='Predicted Label')

    # Loop over data dimensions and create text annotations.
    fmt = '.2f'
    thresh = cm_norm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, f"{format(cm_norm[i, j], fmt)}\n({cm[i, j]})",
                    ha="center", va="center",
                    color="white" if cm_norm[i, j] > thresh else "black")
            
    fig.tight_layout()
    plt.savefig(save_path, dpi=300)
    plt.close()


## Results

In [10]:

CHECKPOINT_PATH = './lightning_logs/first_model/checkpoints/vad-ami-epoch=01-val_f1=0.96.ckpt' 
AUDIO_DIR = './pyannote/amicorpus'
TEST_RTTM_DIR = './only_words/rttms/test'  # Standard AMI test split path

# Run evaluation
evaluate_model(
    checkpoint_path=CHECKPOINT_PATH,
    audio_dir=AUDIO_DIR,
    test_rttm_dir=TEST_RTTM_DIR,
    batch_size=64,
    device='cuda'  # Change to 'cpu' or 'mps' if not running on Nvidia GPU
)

Using device: cuda
Loading model from checkpoint: ./lightning_logs/first_model/checkpoints/vad-ami-epoch=01-val_f1=0.96.ckpt
Preparing test dataset...
Total test chunks to evaluate: 32600

Running evaluation...


Testing: 100%|██████████| 510/510 [00:17<00:00, 28.51it/s]



      VAD EVALUATION REPORT      
Frame-level Accuracy:  0.9391
Precision (Speech):    0.9497
Recall (Sensitivity):  0.9760
F1-Score:              0.9627
AUROC:                 0.9794
Confusion Matrix plot saved as 'confusion_matrix.png'


# Second model

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path 
# Імпортуйте ваші класи (якщо вони в іншому файлі, змініть імпорт)
# з вашого_файлу import SpeakerLightningModule, AMISpeakerDataModule

# from first_model import VADLightningModule, AMIDataModule
from second_model import AMISegmentationTask, AMISegmentationDataModule
from third_model import SpeakerLightningModule, AMISpeakerDataModule

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
C:\Users\gaszt\miniconda3\envs\NLP\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [2]:
import os
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
import torchmetrics
from torchmetrics.classification import BinaryAccuracy, BinaryPrecision, BinaryRecall, BinaryF1Score

# Import your classes from the original training script
from second_model import AMISegmentationDataset, SpeakerSegmentationModel, AMISegmentationTask, PermutationInvariantBCELoss

In [3]:

def run_evaluation(checkpoint_path, audio_dir, test_rttm_dir, batch_size=32, window_size=2.0, window_hop=1.0):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 1. Load the trained Lightning Module
    print("Instantiating model architecture...")
    # Make sure these constants match what you used during training
    base_model = SpeakerSegmentationModel(num_speakers=4, sample_rate=16000)

    print("Loading model checkpoint...")
    # Pass the base_model explicitly to satisfy the __init__ requirement
    task = AMISegmentationTask.load_from_checkpoint(checkpoint_path, model=base_model)
    
    model = task.model
    model.to(device)
    model.eval()

    # 2. Setup Test Dataset and Dataloader
    print("Preparing test dataset...")
    test_dataset = AMISegmentationDataset(
        audio_dir=audio_dir,
        rttm_dir=test_rttm_dir,
        window_size=window_size,
        window_hop=window_hop,
        max_speakers=task.num_speakers
    )
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

    # 3. Initialize evaluation metrics
    # Global frame-level metrics
    test_accuracy = BinaryAccuracy().to(device)
    test_precision = BinaryPrecision().to(device)
    test_recall = BinaryRecall().to(device)
    test_f1 = BinaryF1Score().to(device)

    # Per-speaker slot tracking (Macro breakdown)
    speaker_f1s = [BinaryF1Score().to(device) for _ in range(task.num_speakers)]

    # Diarization-oriented frame counters
    total_speech_frames = 0
    missed_speech_frames = 0
    false_alarm_frames = 0

    # Initialize loss evaluator to perform PIT alignment during inference
    pit_loss_fn = PermutationInvariantBCELoss(num_speakers=task.num_speakers)

    print(f"Starting evaluation on {len(test_dataset)} chunks...")
    
    # 4. Evaluation Loop
    with torch.no_grad():
        for waveforms, targets in tqdm(test_loader, desc="Evaluating"):
            waveforms = waveforms.to(device)
            targets = targets.to(device)

            # Forward pass
            preds = model(waveforms)

            # Interpolate if lengths differ due to pooling
            if preds.size(1) != targets.size(1):
                preds = F.interpolate(preds.permute(0, 2, 1), size=targets.size(1), mode='linear').permute(0, 2, 1)

            # CRITICAL: Use PIT to align targets with the model's channel outputs
            _, aligned_targets = pit_loss_fn(preds, targets)
            preds_prob = torch.sigmoid(preds)

            # Flatten for global metrics
            preds_flat = preds_prob.reshape(-1)
            targets_flat = aligned_targets.reshape(-1)

            # Update global metrics
            test_accuracy.update(preds_flat, targets_flat)
            test_precision.update(preds_flat, targets_flat)
            test_recall.update(preds_flat, targets_flat)
            test_f1.update(preds_flat, targets_flat)

            # Update per-speaker slot metrics
            for spk_idx in range(task.num_speakers):
                speaker_f1s[spk_idx].update(preds_prob[:, :, spk_idx].reshape(-1), aligned_targets[:, :, spk_idx].reshape(-1))

            # Calculate Diarization Error components at frame level
            preds_bool = (preds_prob > 0.5)
            targets_bool = (aligned_targets == 1.0)

            total_speech_frames += torch.sum(targets_bool).item()
            missed_speech_frames += torch.sum(targets_bool & ~preds_bool).item()
            false_alarm_frames += torch.sum(~targets_bool & preds_bool).item()

    # 5. Compute Final Metrics
    avg_acc = test_accuracy.compute().item()
    avg_prec = test_precision.compute().item()
    avg_rec = test_recall.compute().item()
    avg_f1 = test_f1.compute().item()

    spk_f1_scores = [metric.compute().item() for metric in speaker_f1s]
    macro_f1 = sum(spk_f1_scores) / len(spk_f1_scores)

    # Diarization error rates normalized by total true active speech frames
    miss_rate = (missed_speech_frames / total_speech_frames) if total_speech_frames > 0 else 0.0
    fa_rate = (false_alarm_frames / total_speech_frames) if total_speech_frames > 0 else 0.0
    
    # In a perfectly PIT-aligned scenario, Speaker Confusion transitions into Miss/FA.
    # This sum serves as a highly accurate frame-level Diarization Error Rate (DER) proxy.
    frame_der_proxy = miss_rate + fa_rate 

    # 6. Print Professional Summary Report
    print("\n" + "="*50)
    print("          SPEAKER SEGMENTATION TEST REPORT          ")
    print("="*50)
    print(f"Global Accuracy:  {avg_acc * 100:.2f}%")
    print(f"Global Precision: {avg_prec * 100:.2f}%")
    print(f"Global Recall:    {avg_rec * 100:.2f}%")
    print(f"Global F1-Score:  {avg_f1 * 100:.2f}%")
    print(f"Macro F1-Score:   {macro_f1 * 100:.2f}%")
    print("-"*50)
    print("Per-Speaker Output Slot Performance (F1-Score):")
    for i, score in enumerate(spk_f1_scores):
        print(f"  • Speaker Slot {i}: {score * 100:.2f}%")
    print("-"*50)
    print("Diarization-Inspired Frame Analysis:")
    print(f"  • Missed Speech Rate: {miss_rate * 100:.2f}% (True speech missed by model)")
    print(f"  • False Alarm Rate:   {fa_rate * 100:.2f}% (Silence predicted as speech)")
    print(f"  • Frame-level DER Proxy: {frame_der_proxy * 100:.2f}%")
    print("="*50 + "\n")


## Results

In [4]:
CHECKPOINT_PATH = "./lightning_logs/second_model/checkpoints/epoch=1-step=18140.ckpt" # Adjust to your actual path
AUDIO_DIR = './pyannote/amicorpus'
TEST_RTTM_DIR = './only_words/rttms/test'

if os.path.exists(CHECKPOINT_PATH):
    run_evaluation(
        checkpoint_path=CHECKPOINT_PATH,
        audio_dir=AUDIO_DIR,
        test_rttm_dir=TEST_RTTM_DIR
    )
else:
    print(f"Error: Checkpoint file not found at '{CHECKPOINT_PATH}'. Please update the path.")

Using device: cuda
Instantiating model architecture...
Loading model checkpoint...


F:\PythonProjects\AMI-diarization-setup\second_model.py:73: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  info = torchaudio.info(audio_path)
C:\Users\gaszt\AppData\Roaming\Python\Python311\site-packages\torchaudio\_backend\soundfile_backend.py:120: UserWarning: torchaudio._backend.common.AudioMetaData has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be remo

Preparing test dataset...
Starting evaluation on 32600 chunks...


Evaluating:   0%|          | 0/1019 [00:00<?, ?it/s]C:\Users\gaszt\AppData\Roaming\Python\Python311\site-packages\torchaudio\_backend\utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
Evaluating: 100%|██████████| 1019/1019 [01:05<00:00, 15.61it/s]


          SPEAKER SEGMENTATION TEST REPORT          
Global Accuracy:  95.47%
Global Precision: 93.21%
Global Recall:    87.10%
Global F1-Score:  90.05%
Macro F1-Score:   40.88%
--------------------------------------------------
Per-Speaker Output Slot Performance (F1-Score):
  • Speaker Slot 0: 2.32%
  • Speaker Slot 1: 0.00%
  • Speaker Slot 2: 94.99%
  • Speaker Slot 3: 66.23%
--------------------------------------------------
Diarization-Inspired Frame Analysis:
  • Missed Speech Rate: 12.90% (True speech missed by model)
  • False Alarm Rate:   6.34% (Silence predicted as speech)
  • Frame-level DER Proxy: 19.24%



# Third model


In [2]:

def run_similarity_test(checkpoint_path, audio_dir, val_rttm_dir, batch_size=32):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    # 1. Ініціалізація DataModule для отримання словника мовців та завантажувача даних
    print("Preparing data...")
    # Трейнінг RTTM передаємо як None, бо для тесту нам потрібен лише валідаційний/тестовий сет
    data_module = AMISpeakerDataModule(
        audio_dir=audio_dir,
        train_rttm_dir=Path('./'), 
        val_rttm_dir=val_rttm_dir,
        batch_size=batch_size,
        num_workers=0
    )
    # Ініціалізуємо датасет вручну через setup
    data_module.setup(stage="validate")
    val_loader = data_module.val_dataloader()

    # 2. Завантаження навченої моделі з чекпоінту
    print(f"Loading checkpoint from: {checkpoint_path}")
    model = SpeakerLightningModule.load_from_checkpoint(checkpoint_path)
    model.to(device)
    model.eval()  # Переводимо в режим оцінювання (вимкнення dropout, batchnorm у режим тесту)

    all_embeddings = []
    all_labels = []

    # 3. Екстракція ембедінгів для всієї вибірки
    print("Extracting speaker embeddings...")
    with torch.no_grad():
        for batch in tqdm(val_loader):
            waveforms, labels = batch
            waveforms = waveforms.to(device)

            # Отримуємо L2-нормалізовані ембедінги
            embeddings = model(waveforms)

            all_embeddings.append(embeddings.cpu())
            all_labels.append(labels)

            
    # Об'єднуємо всі батчі в єдині тензори
    all_embeddings = torch.cat(all_embeddings, dim=0)  # [N, 256]
    all_labels = torch.cat(all_labels, dim=0)          # [N]
    
    num_samples = all_embeddings.size(0)
    print(f"Extracted {num_samples} audio segments.")

    # 4. Обчислення косинусної подібності
    # Оскільки ембедінги вже L2-нормалізовані в моделі, матричне множення дає саме Cosine Similarity
    print("Computing pairwise cosine similarity matrix...")
    similarity_matrix = torch.mm(all_embeddings, all_embeddings.t()) # [N, N]

    # 5. Створення масок для фільтрації результатів
    # Матриця, де True означає, що мовці однакові
    same_speaker_mask = (all_labels.unsqueeze(0) == all_labels.unsqueeze(1))
    
    # Виключаємо діагональ (порівняння аудіо-сегмента з самим собою, де схожість завжди 1.0)
    diag_mask = torch.eye(num_samples, dtype=torch.bool)
    same_speaker_mask = same_speaker_mask & ~diag_mask
    diff_speaker_mask = ~same_speaker_mask & ~diag_mask

    # Витягуємо значення у вигляді одновимірних векторів (numpy для зручності аналізу)
    same_speaker_sims = similarity_matrix[same_speaker_mask].numpy()
    diff_speaker_sims = similarity_matrix[diff_speaker_mask].numpy()

    # 6. Розрахунок та вивід статистичних метрик
    print("\n" + "="*40)
    print("  COSINE SIMILARITY TEST RESULTS  ")
    print("="*40)
    
    if len(same_speaker_sims) > 0:
        print(f"Same Speaker (Positive Pairs) - Total: {len(same_speaker_sims)}")
        print(f"  Mean Similarity:   {np.mean(same_speaker_sims):.4f}")
        print(f"  Median Similarity: {np.median(same_speaker_sims):.4f}")
        print(f"  Std Deviation:     {np.std(same_speaker_sims):.4f}")
        print(f"  Min Similarity:    {np.min(same_speaker_sims):.4f}")
        print(f"  Max Similarity:    {np.max(same_speaker_sims):.4f}")
    else:
        print("No same speaker pairs found in the batch/dataset!")

    print("-" * 40)
    
    if len(diff_speaker_sims) > 0:
        print(f"Different Speakers (Negative Pairs) - Total: {len(diff_speaker_sims)}")
        print(f"  Mean Similarity:   {np.mean(diff_speaker_sims):.4f}")
        print(f"  Median Similarity: {np.median(diff_speaker_sims):.4f}")
        print(f"  Std Deviation:     {np.std(diff_speaker_sims):.4f}")
        print(f"  Min Similarity:    {np.min(diff_speaker_sims):.4f}")
        print(f"  Max Similarity:    {np.max(diff_speaker_sims):.4f}")
    else:
        print("No different speaker pairs found!")
    print("="*40)

    # 7. Візуалізація розподілу (Побудова НОРМОВАНОЇ гістограми)
    plt.figure(figsize=(10, 6))
    
    # Додаємо density=True для відображення щільності ймовірності замість кількості
    if len(same_speaker_sims) > 0:
        plt.hist(same_speaker_sims, bins=50, alpha=0.6, density=True, 
                 label='Same Speaker (Positives)', color='g', edgecolor='black')
                 
    if len(diff_speaker_sims) > 0:
        plt.hist(diff_speaker_sims, bins=50, alpha=0.6, density=True, 
                 label='Different Speakers (Negatives)', color='r', edgecolor='black')
        
    plt.title('Normalized Distribution of Cosine Similarities')
    plt.xlabel('Cosine Similarity')
    plt.ylabel('Density (Probability)') # Змінено назву осі Y
    plt.legend(loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Зберігаємо графік у файл
    output_plot = "speaker_similarity_distribution_normalized.png"
    plt.savefig(output_plot)
    print(f"Normalized distribution plot saved as '{output_plot}'")
    plt.show()


## Results

In [3]:
CHECKPOINT_PATH = "./lightning_logs/model_three/checkpoints/epoch=9-step=42660.ckpt" 
AUDIO_DIR = './pyannote/amicorpus'
VAL_RTTM_DIR = './only_words/rttms/test'

run_similarity_test(
    checkpoint_path=CHECKPOINT_PATH,
    audio_dir=AUDIO_DIR,
    val_rttm_dir=VAL_RTTM_DIR,
    batch_size=32
)

Using device: cuda
Preparing data...


FileNotFoundError: [Errno 2] No such file or directory: 'EN2001a.rttm'